In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("demandsense_consistent.csv")
df["date"] = pd.to_datetime(df["date"])

hotel_df = df[df["hotel_id"] == "H001"].copy()
hotel_df = hotel_df.sort_values("date").reset_index(drop=True)

print("Dataset shape:", hotel_df.shape)
print(hotel_df.head())

Dataset shape: (1381, 21)
        date hotel_id           hotel_name  total_rooms  available_rooms  \
0 2023-01-01     H001  Mountain View Hotel           80               26   
1 2023-01-02     H001  Mountain View Hotel           80               42   
2 2023-01-03     H001  Mountain View Hotel           80               43   
3 2023-01-04     H001  Mountain View Hotel           80               44   
4 2023-01-05     H001  Mountain View Hotel           80               35   

   occupied_rooms  occupancy_rate  bookings  bookings_cancelled  avg_price  \
0              54           67.50        36                   7    4747.33   
1              38           47.50        41                   5    4701.70   
2              37           46.25        41                   3    4172.83   
3              36           45.00        34                   2    4253.72   
4              45           56.25        38                   2    4290.55   

   ... month  is_weekend  is_holiday  festival_o

In [11]:
hotel_df["occupancy_lag_1"] = hotel_df["occupancy_rate"].shift(1)

hotel_df["occupancy_lag_7"] = (
    hotel_df["occupancy_rate"].shift(1).rolling(window=7).mean()
)

hotel_df = hotel_df.dropna().reset_index(drop=True)

print(hotel_df[
    ["date", "occupancy_rate", "occupancy_lag_1", "occupancy_lag_7"]
].head())

        date  occupancy_rate  occupancy_lag_1  occupancy_lag_7
0 2023-01-08           67.50            57.50        51.428571
1 2023-01-09           41.25            67.50        51.428571
2 2023-01-10           47.50            41.25        50.535714
3 2023-01-11           31.25            47.50        50.714286
4 2023-01-12           35.00            31.25        48.750000


In [12]:
features = [
    "month",
    "day",
    "day_of_year",
    "is_weekend",
    "is_holiday",
    "occupancy_lag_1",
    "occupancy_lag_7"
]

X = hotel_df[features]
y = hotel_df["occupancy_rate"]

split_index = int(len(hotel_df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 1099
Testing rows: 275


In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("Random Forest MAE:", round(mae, 2))
print("Random Forest RMSE:", round(rmse, 2))

Random Forest MAE: 6.09
Random Forest RMSE: 7.6


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Baseline: predict today's occupancy using the previous day's occupancy
baseline_predictions = X_test["occupancy_lag_1"]

baseline_mae = mean_absolute_error(y_test, baseline_predictions)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)

print("Model Comparison")
print("----------------")
print("Random Forest MAE:", round(mae, 2))
print("Baseline MAE:", round(baseline_mae, 2))

print("Random Forest RMSE:", round(rmse, 2))
print("Baseline RMSE:", round(baseline_rmse, 2))

Model Comparison
----------------
Random Forest MAE: 6.09
Baseline MAE: 11.02
Random Forest RMSE: 7.6
Baseline RMSE: 13.97


In [15]:
%pip install xgboost


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, y_train)

xgb_predictions = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(
    mean_squared_error(y_test, xgb_predictions)
)

print("XGBoost MAE:", round(xgb_mae, 2))
print("XGBoost RMSE:", round(xgb_rmse, 2))


XGBoost MAE: 5.73
XGBoost RMSE: 7.2


In [19]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, y_train)

xgb_predictions = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(
    mean_squared_error(y_test, xgb_predictions)
)

print("XGBoost MAE:", round(xgb_mae, 2))
print("XGBoost RMSE:", round(xgb_rmse, 2))

XGBoost MAE: 5.73
XGBoost RMSE: 7.2


In [20]:
results = pd.DataFrame({
    "Model": ["Baseline", "Random Forest", "XGBoost"],
    "MAE": [baseline_mae, mae, xgb_mae],
    "RMSE": [baseline_rmse, rmse, xgb_rmse]
})

results = results.sort_values("MAE").reset_index(drop=True)

print("Final Model Comparison")
print("======================")
print(results.round(2).to_string(index=False))

Final Model Comparison
        Model   MAE  RMSE
      XGBoost  5.73  7.20
Random Forest  6.09  7.60
     Baseline 11.02 13.97


In [ ]:
!streamlit run app.py
